# L7 - Agent optimization
## What you'll learn
- How to ask AgentCore Optimization for a better system prompt for the Orchestrator Harness
- How to ask for better tool descriptions for the Refund Agent
- How to save the suggestions as versioned configuration bundles
## Overview
AgentCore Optimization looks at the real conversations of your agents and suggests ways to improve them. You save each suggestion as a versioned configuration bundle. A bundle works like a git commit for the settings of your agent, so you always have a history to look back at.

In this notebook you:
- Have a few conversations with the Harness, so there are traces to learn from.
- Ask for a better system prompt and compare it with the current one.
- Ask for better tool descriptions for the Refund Agent.
- Save the current and the recommended settings as two versions of a bundle.
## Architecture
![Agent operations layer: spans from the agents in CloudWatch feed AgentCore Evaluations, which produces session, trace and tool scores, and AgentCore Optimization, which produces versioned configuration bundles.](images/l7-agent-operations-architecture.png)

This notebook covers the Optimization half of the diagram. It reads the traces of the agents from CloudWatch and produces versioned configuration bundles.
## Steps
1. Set up access and load the workshop settings
2. Have a few conversations with the Harness
3. Ask for a better system prompt
4. Ask for better tool descriptions on the Refund Agent
5. Save the current settings as version 1
6. Save the recommended prompt as version 2
7. Look at the version history
8. Clean up

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites
- Complete the L1 to L6 notebooks, ending with [`1_cloudwatch_omni.ipynb`](../l6-observability/1_cloudwatch_omni.ipynb), so the agents are deployed and have written traces to CloudWatch.
- The agent evaluations notebook ([`1_agent_evaluations.ipynb`](1_agent_evaluations.ipynb)) is not required, but it helps to read it first.

**Cost note:** the whole notebook makes a handful of Bedrock model calls and costs well under $1.

> **Note:** AgentCore Optimization reads traces from CloudWatch only, so it doesn't work if the agents send their traces to any other observability provider like Datadog.

## Step 1: Set up access and load the workshop settings

Install the packages used in this notebook. If they are already installed, this does nothing.

In [ ]:
%pip install --quiet --upgrade boto3 'pandas==2.2.3'

Load the libraries, connect to AWS and read the workshop settings saved by earlier notebooks.

In [ ]:
import boto3
import json
import os
import time
import uuid
from datetime import datetime, timedelta, timezone

import pandas as pd

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

# Two AgentCore clients:
#   agentcore          — runs things (invoke, evaluate, get recommendations, ...)
#   agentcore_control  — manages resources (harnesses, evaluators, bundles, ...)
ssm = boto3.client("ssm", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)
iam = boto3.client("iam", region_name=REGION)
logs_client = boto3.client("logs", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

SSM_PREFIX = "/anycompany/agentcore"
ACCOUNT_ID = sts.get_caller_identity()["Account"]

# Load a few settings from Parameter Store
cfg = {}
for k in ["harness_id", "harness_arn", "harness_name", "gateway_arn", "cognito_client_id", "user_password", "refund_agent_runtime_arn"]:
    try:
        cfg[k] = ssm.get_parameter(Name="{}/{}".format(SSM_PREFIX, k), WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        cfg[k] = None
        print("  Warning: missing {}/{}".format(SSM_PREFIX, k))

if not cfg.get("harness_arn"):
    raise RuntimeError("Harness ARN not found in SSM. Run L3 notebook #5 (3_orchestrator_agent.ipynb) first.")

print("Region:  {}".format(REGION))
print("Account: {}".format(ACCOUNT_ID))
print("Harness: {}".format(cfg["harness_arn"]))

Find the CloudWatch log group where the Harness writes its traces, so later steps can point AgentCore at it.

In [ ]:
HARNESS_LOG_GROUP = None
HARNESS_SERVICE_NAME = None
prefix = "/aws/bedrock-agentcore/runtimes/harness_{}-".format(cfg["harness_name"])
for lg in logs_client.describe_log_groups(logGroupNamePrefix=prefix).get("logGroups", []):
    HARNESS_LOG_GROUP = lg["logGroupName"]
    # Turn the log group name into the OpenTelemetry service name AgentCore expects.
    tail = HARNESS_LOG_GROUP.rsplit("/", 1)[-1]
    endpoint = tail.rsplit("-", 1)[-1]
    base = tail.rsplit("-", 1)[0].rsplit("-", 1)[0]
    HARNESS_SERVICE_NAME = "{}.{}".format(base, endpoint)
    break

if not HARNESS_LOG_GROUP:
    raise RuntimeError("Couldn't find the Harness log group. Was L3 notebook #5 (3_orchestrator_agent.ipynb) deployed with observability?")

# The recommendation API needs the log group's full ARN, not just the name.
HARNESS_LOG_GROUP_ARN = "arn:aws:logs:{}:{}:log-group:{}".format(REGION, ACCOUNT_ID, HARNESS_LOG_GROUP)

print("Log group:    {}".format(HARNESS_LOG_GROUP))
print("Service name: {}".format(HARNESS_SERVICE_NAME))

Read the current settings of the Harness. This gives us its current system prompt, which we want to improve.

In [ ]:
resp = agentcore_control.get_harness(harnessId=cfg["harness_id"])
harness_state = resp["harness"]

# Flatten the system prompt (it comes back as a list of pieces)
CURRENT_SYSTEM_PROMPT = "\n".join(
    seg.get("text", "") for seg in harness_state.get("systemPrompt", []) if isinstance(seg, dict)
)
if not CURRENT_SYSTEM_PROMPT:
    raise RuntimeError("The Harness has no system prompt \u2014 nothing to optimize.")

print("Current system prompt: {} characters".format(len(CURRENT_SYSTEM_PROMPT)))
print(CURRENT_SYSTEM_PROMPT)

Give AgentCore Optimization permission to read your traces and call the models it needs. This is the same role the evaluations notebook uses, so if that notebook ran, this cell only refreshes the permissions.

In [ ]:
OPT_ROLE_NAME = "AgentCoreEvaluationRole"

TRUST_POLICY = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole",
        "Condition": {"StringEquals": {
            "aws:SourceAccount": ACCOUNT_ID,
            "aws:ResourceAccount": ACCOUNT_ID,
        }}
    }]
}

INLINE_POLICY = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "CloudWatchLogsRead",
            "Effect": "Allow",
            "Action": [
                "logs:DescribeLogGroups", "logs:GetQueryResults", "logs:StartQuery",
                "logs:GetLogEvents", "logs:FilterLogEvents",
            ],
            "Resource": "*",
        },
        {
            "Sid": "CloudWatchLogsWrite",
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "arn:aws:logs:{}:{}:log-group:/aws/bedrock-agentcore/*".format(REGION, ACCOUNT_ID),
        },
        {
            "Sid": "CloudWatchIndex",
            "Effect": "Allow",
            "Action": ["logs:DescribeIndexPolicies", "logs:PutIndexPolicy"],
            "Resource": [
                "arn:aws:logs:{}:{}:log-group:aws/spans".format(REGION, ACCOUNT_ID),
                "arn:aws:logs:{}:{}:log-group:aws/spans:*".format(REGION, ACCOUNT_ID),
            ],
        },
        {
            "Sid": "BedrockInvoke",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream",
                "bedrock:Converse", "bedrock:ConverseStream",
            ],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                "arn:aws:bedrock:*:{}:inference-profile/*".format(ACCOUNT_ID),
                "arn:aws:bedrock:*:{}:application-inference-profile/*".format(ACCOUNT_ID),
            ],
        },
    ]
}

try:
    iam.create_role(RoleName=OPT_ROLE_NAME,
                    AssumeRolePolicyDocument=json.dumps(TRUST_POLICY),
                    Description="AgentCore optimization + evaluations service role")
    print("Created role: {}".format(OPT_ROLE_NAME))
except iam.exceptions.EntityAlreadyExistsException:
    iam.update_assume_role_policy(RoleName=OPT_ROLE_NAME, PolicyDocument=json.dumps(TRUST_POLICY))
    print("Role already exists, refreshed its trust policy: {}".format(OPT_ROLE_NAME))

iam.put_role_policy(RoleName=OPT_ROLE_NAME,
                    PolicyName="AgentCoreOptimizationInline",
                    PolicyDocument=json.dumps(INLINE_POLICY))
OPT_ROLE_ARN = iam.get_role(RoleName=OPT_ROLE_NAME)["Role"]["Arn"]
print("Role ARN: {}".format(OPT_ROLE_ARN))

# AWS needs a few seconds to make the new permissions available to other services
print("Waiting 10 seconds for the role to be usable...")
time.sleep(10)

## Step 2: Have a few conversations with the Harness
Optimization can only suggest improvements from real conversations. This step chats with the Harness twice, using prompts that are known to be hard for the current system prompt: one ambiguous request and one that needs several tools.

Each conversation creates traces in CloudWatch. After the requests, wait 2-3 minutes for the traces to be saved.

In [ ]:
# Two prompts — enough to give the recommendation service data to work with,
# without ballooning the workshop wall time. The first is deliberately ambiguous
# (order vs refund); the second exercises multi-tool routing.
OPT_PROMPTS = [
    "I'm not sure what's going on with my order. Something's wrong. Please help.",
    "ORD-10001 was supposed to arrive last week and I haven't got it. Can you check and refund me?",
]

trace_sessions = []
for i, p in enumerate(OPT_PROMPTS, 1):
    session_id = str(uuid.uuid4()).upper()
    try:
        resp = agentcore.invoke_harness(
            harnessArn=cfg["harness_arn"],
            runtimeSessionId=session_id,
            actorId="CUST-789",
            messages=[{"role": "user", "content": [{"text": p}]}],
        )
        # Read all the streaming response chunks so the conversation finishes cleanly
        for _ in resp["stream"]:
            pass
        trace_sessions.append(session_id)
        print("[{}/{}] {} \u2014 session={}".format(i, len(OPT_PROMPTS), p[:55], session_id))
    except Exception as e:
        print("[{}/{}] {} \u2014 FAILED: {}".format(i, len(OPT_PROMPTS), p[:55], e))

# Wait for the trace data to arrive in CloudWatch.
INGEST_WAIT = 180
print("\nWaiting {} seconds for AWS to save the trace data...".format(INGEST_WAIT))
for remaining in range(INGEST_WAIT, 0, -30):
    print("  {} seconds remaining".format(remaining))
    time.sleep(30)
print("Done.")

## Step 3: Ask for a better system prompt
Ask AgentCore Optimization to read those conversations and suggest a better system prompt. You give it:
1. The prompt you use now, so it has something to compare against.
2. The traces to look at, which are the Harness log group and a time window.
3. The quality measure to improve. Here it is goal success, which checks whether the agent completed the goal of the user.

An AI model writes the improved prompt in the background.

In [ ]:
SYSPROMPT_REC_NAME = "harness_sysprompt_inline_{}".format(uuid.uuid4().hex[:6])

# Fetch spans for the sessions we generated in Step 2
def fetch_session_spans(log_groups, session_ids, wait_seconds=60):
    all_spans = []
    end_ = datetime.now(timezone.utc)
    start_ = end_ - timedelta(hours=2)

    def _one_query(log_group, session_id):
        query = (
            'fields @timestamp, @message'
            '| filter ispresent(scope.name) and ispresent(attributes.session.id)'
            '| filter attributes.session.id = "{}"'
            '| sort @timestamp asc'
        ).format(session_id)
        try:
            q_id = logs_client.start_query(
                logGroupName=log_group,
                startTime=int(start_.timestamp()),
                endTime=int(end_.timestamp()),
                queryString=query,
            )["queryId"]
        except logs_client.exceptions.ResourceNotFoundException:
            return []
        deadline = time.time() + wait_seconds
        r = {"status": "Running"}
        while time.time() < deadline:
            r = logs_client.get_query_results(queryId=q_id)
            if r["status"] in ("Complete", "Failed"):
                break
            time.sleep(1)
        if r.get("status") != "Complete":
            return []
        out = []
        for row in r.get("results", []):
            for f in row:
                if f["field"] == "@message" and f["value"].strip().startswith("{"):
                    try:
                        out.append(json.loads(f["value"]))
                    except json.JSONDecodeError:
                        pass
        return out

    for lg in log_groups:
        for sid in session_ids:
            all_spans.extend(_one_query(lg, sid))
    return all_spans

# The recommendation API accepts up to 1000 spans per call
SESSIONSPANS_MAX = 1000

def cap_spans_fairly(spans, session_ids):
    if len(spans) <= SESSIONSPANS_MAX:
        return spans
    per_session = SESSIONSPANS_MAX // max(len(session_ids), 1)
    buckets = {sid: [] for sid in session_ids}
    for s in spans:
        sid = (s.get("attributes", {}) or {}).get("session.id")
        if sid in buckets:
            buckets[sid].append(s)
    out = []
    for sid in session_ids:
        out.extend(buckets[sid][:per_session])
    return out[:SESSIONSPANS_MAX]

print("Fetching spans for {} sessions from Step 2...".format(len(trace_sessions)))
harness_spans = fetch_session_spans(["aws/spans", HARNESS_LOG_GROUP], trace_sessions)
print("  Downloaded {} spans".format(len(harness_spans)))

# Wait for CloudWatch trace ingestion
if not harness_spans:
    print("  No spans yet \u2014 waiting another 60s for ingestion to catch up...")
    time.sleep(60)
    harness_spans = fetch_session_spans(["aws/spans", HARNESS_LOG_GROUP], trace_sessions)
    print("  Downloaded {} spans (retry)".format(len(harness_spans)))

if not harness_spans:
    raise RuntimeError(
        "Still no spans for Step 2 sessions. Wait another minute and re-run this cell, "
        "or check aws/spans in the CloudWatch console for session.id values matching " + 
        str(trace_sessions[:2]))

# Cap at 1000 spans if needed (API limit). Preserves whole sessions.
raw_count = len(harness_spans)
harness_spans = cap_spans_fairly(harness_spans, trace_sessions)
print("  Sending {} spans (from {} raw) to the recommendation service.".format(len(harness_spans), raw_count))

resp = agentcore.start_recommendation(
    name=SYSPROMPT_REC_NAME,
    description="Optimize the AnyCompany orchestrator's system prompt for GoalSuccessRate.",
    type="SYSTEM_PROMPT_RECOMMENDATION",
    recommendationConfig={
        "systemPromptRecommendationConfig": {
            # 1. The current prompt (what we want to improve)
            "systemPrompt": {"text": CURRENT_SYSTEM_PROMPT},
            # 2. Exact traces to analyze (only our fresh sessions, inline)
            "agentTraces": {"sessionSpans": harness_spans},
            # 3. What "better" means — optimize for goal success rate
            "evaluationConfig": {
                "evaluators": [
                    {"evaluatorArn": "arn:aws:bedrock-agentcore:::evaluator/Builtin.GoalSuccessRate"},
                ]
            },
        }
    },
)
SYSPROMPT_REC_ID = resp["recommendationId"]
print("\nStarted recommendation: {}".format(SYSPROMPT_REC_ID))
print("Status: {}".format(resp["status"]))

The recommendation runs in the background. Wait for it to finish. The cell prints the elapsed time every 20 seconds.

In [ ]:
# The recommendation runs asynchronously. Poll until it reaches a terminal state.
def poll_recommendation(rec_id, timeout_minutes=20):
    start = time.time()
    deadline = start + timeout_minutes * 60
    while time.time() < deadline:
        r = agentcore.get_recommendation(recommendationId=rec_id)
        status = r["status"]
        elapsed = int(time.time() - start)
        print("  [{:>3}s] status={}".format(elapsed, status))
        if status in ("COMPLETED", "FAILED", "DELETING"):
            return r
        time.sleep(20)
    print("  Timeout reached at {} min. Returning latest status anyway.".format(timeout_minutes))
    return agentcore.get_recommendation(recommendationId=rec_id)

result = poll_recommendation(SYSPROMPT_REC_ID)
print("\nFinal status: {}".format(result["status"]))

# Pull out the recommended prompt and explanation
sp_result = (result.get("recommendationResult") or {}).get("systemPromptRecommendationResult") or {}
if sp_result.get("errorCode"):
    print("ERROR: {} \u2014 {}".format(sp_result.get("errorCode"), sp_result.get("errorMessage")))
    RECOMMENDED_SYSTEM_PROMPT = None
else:
    RECOMMENDED_SYSTEM_PROMPT = sp_result.get("recommendedSystemPrompt")
    print("Got a recommendation ({} characters).".format(len(RECOMMENDED_SYSTEM_PROMPT or "")))

Compare the current prompt with the recommended one side by side, and read the explanation of what changed and why.

In [ ]:
if not RECOMMENDED_SYSTEM_PROMPT:
    print("(No recommendation to show \u2014 see error above.)")
else:
    print("=" * 20 + " CURRENT PROMPT " + "=" * 20)
    print(CURRENT_SYSTEM_PROMPT[:1000])
    if len(CURRENT_SYSTEM_PROMPT) > 1000:
        print("... ({} chars total)".format(len(CURRENT_SYSTEM_PROMPT)))

    print("\n" + "=" * 20 + " RECOMMENDED PROMPT " + "=" * 20)
    print(RECOMMENDED_SYSTEM_PROMPT[:1000])
    if len(RECOMMENDED_SYSTEM_PROMPT) > 1000:
        print("... ({} chars total)".format(len(RECOMMENDED_SYSTEM_PROMPT)))

    print("\n" + "=" * 20 + " WHY THIS CHANGE " + "=" * 20)
    print(sp_result.get("explanation") or "(no explanation returned)")

## Step 4: Ask for better tool descriptions on the Refund Agent
The agent reads the description of each tool to decide which one to call. Vague descriptions can make it pick the wrong tool.

The Refund Agent has three tools: one that looks up the refund policy, one that checks whether an order qualifies for a refund, and one that issues the refund. Sharper descriptions help the agent choose between them.

This step:
1. Finds the Refund Agent and its log group.
2. Signs in as `gold_customer`.
3. Sends three prompts that use all three tools.
4. Waits for the traces to be saved.
5. Asks AgentCore Optimization to sharpen the tool descriptions.

Find the Refund Agent and its log group, and sign in so you can talk to it.

In [ ]:
import requests
from urllib.parse import quote
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

# 1. Read the Refund Agent's runtime ARN from SSM (published by L3 notebook #3)
REFUND_RUNTIME_ARN = cfg.get("refund_agent_runtime_arn")
if not REFUND_RUNTIME_ARN:
    raise RuntimeError("Refund Agent runtime ARN not found in SSM. Was L3 notebook #3 deployed?")

# 2. Derive log group + service name (same pattern as the Harness)
REFUND_RUNTIME_ID = REFUND_RUNTIME_ARN.split("/")[-1]
REFUND_LOG_GROUP = "/aws/bedrock-agentcore/runtimes/{}-DEFAULT".format(REFUND_RUNTIME_ID)
REFUND_LOG_GROUP_ARN = "arn:aws:logs:{}:{}:log-group:{}".format(REGION, ACCOUNT_ID, REFUND_LOG_GROUP)
REFUND_SERVICE_NAME = "{}.DEFAULT".format(REFUND_RUNTIME_ID.rsplit("-", 1)[0])

print("Refund Agent:    {}".format(REFUND_RUNTIME_ARN))
print("Log group:       {}".format(REFUND_LOG_GROUP))
print("Service name:    {}".format(REFUND_SERVICE_NAME))

# 3. Cognito auth for A2A calls — bearer token identifies us as the gold_customer
cognito = boto3.client("cognito-idp", region_name=REGION)
auth = cognito.initiate_auth(
    ClientId=cfg["cognito_client_id"],
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": "gold_customer", "PASSWORD": cfg["user_password"]},
)
BEARER_TOKEN = auth["AuthenticationResult"]["IdToken"]
print("\nCognito authentication: got bearer token ({} chars)".format(len(BEARER_TOKEN)))

# 4. A2A invocation helper. Same pattern as L3 notebooks: SigV4 sign the request,
#    then attach the Cognito bearer token as an Authorization header.
def invoke_a2a(runtime_arn, prompt, session_id):
    escaped_arn = quote(runtime_arn, safe="")
    url = "https://bedrock-agentcore.{}.amazonaws.com/runtimes/{}/invocations".format(REGION, escaped_arn)
    payload = json.dumps({
        "jsonrpc": "2.0",
        "method": "message/send",
        "id": str(uuid.uuid4()),
        "params": {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": prompt}],
                "messageId": str(uuid.uuid4()),
            }
        },
    })
    creds = boto3.Session(region_name=REGION).get_credentials().get_frozen_credentials()
    headers = {
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
        "Authorization": "Bearer {}".format(BEARER_TOKEN),
    }
    aws_req = AWSRequest(method="POST", url=url, data=payload, headers=headers)
    SigV4Auth(creds, "bedrock-agentcore", REGION).add_auth(aws_req)
    return requests.post(url, data=payload, headers=dict(aws_req.headers), timeout=180)

print("\nA2A helper ready.")

Send three prompts to the Refund Agent that use all three of its tools. Each conversation creates traces to learn from. Then wait 3 minutes for the traces to be saved.

In [ ]:
# Three prompts — one per Refund Agent tool, so each tool gets exercised at
# least once and appears in the traces for the recommendation service.
REFUND_TOOL_PROMPTS = [
    # Exercises refund_policy (KB lookup)
    "What is your refund policy for electronics?",
    # Exercises check_eligible (uses order metadata)
    "Am I eligible for a refund on ORD-10001, which was delivered but damaged?",
    # Exercises check_eligible + process_refund
    "Please process a $50 refund on ORD-10005 for a defective product.",
]

refund_sessions = []
for i, p in enumerate(REFUND_TOOL_PROMPTS, 1):
    sid = str(uuid.uuid4()).upper()
    try:
        r = invoke_a2a(REFUND_RUNTIME_ARN, p, sid)
        if r.status_code == 200:
            refund_sessions.append(sid)
            print("[{}/{}] {} \u2014 session={}".format(i, len(REFUND_TOOL_PROMPTS), p[:55], sid))
        else:
            print("[{}/{}] FAILED HTTP {} \u2014 {}".format(i, len(REFUND_TOOL_PROMPTS), r.status_code, r.text[:200]))
    except Exception as e:
        print("[{}/{}] FAILED: {}".format(i, len(REFUND_TOOL_PROMPTS), e))

# Wait for trace ingestion
print("\nWaiting {} seconds for AWS to save the trace data...".format(INGEST_WAIT))
for remaining in range(INGEST_WAIT, 0, -30):
    print("  {} seconds remaining".format(remaining))
    time.sleep(30)
print("Done.")

Use the real tool descriptions from the Refund Agent, which is what the agent sees now when it picks a tool.

As in Step 3, pass only the traces of the sessions you just created. They also show which tools were used, so descriptions are improved only for tools with real usage behind them.

In [ ]:
# The Refund Agent's actual tool set + descriptions 
REFUND_AGENT_TOOLS = [
    {
        "toolName": "refund_policy",
        "toolDescription": (
            "Retrieve refund policy information from the knowledge base.\n\n"
            "Args:\n    question: The customer's question about refund policy."
        ),
    },
    {
        "toolName": "check_eligible",
        "toolDescription": (
            "Check whether an order is eligible for a refund.\n\n"
            "Args:\n"
            "    order_id: The order identifier (e.g. ORD-12345).\n"
            "    order_status: Current order status (e.g. delivered, shipped).\n"
            "    item: Name of the product.\n"
            "    reason: Reason for the refund request.\n"
            "    damage_level: Severity of damage if applicable."
        ),
    },
    {
        "toolName": "process_refund",
        "toolDescription": (
            "Process an approved refund for an order.\n\n"
            "Args:\n"
            "    order_id: The order identifier.\n"
            "    refund_amount: Dollar amount to refund.\n"
            "    reason: Reason for the refund.\n"
            "    payment_method: Where to send the refund."
        ),
    },
]

# Fetch spans for the Refund Agent sessions we just generated. 
print("Fetching spans for {} Refund Agent sessions...".format(len(refund_sessions)))
refund_spans = fetch_session_spans(["aws/spans", REFUND_LOG_GROUP], refund_sessions)
print("  Downloaded {} spans".format(len(refund_spans)))
if not refund_spans:
    print("  No spans yet \u2014 waiting another 60s for ingestion to catch up...")
    time.sleep(60)
    refund_spans = fetch_session_spans(["aws/spans", REFUND_LOG_GROUP], refund_sessions)
    print("  Downloaded {} spans (retry)".format(len(refund_spans)))

# Cap at 1000 spans if needed (API limit). Preserves whole sessions.
raw_count = len(refund_spans)
refund_spans = cap_spans_fairly(refund_spans, refund_sessions)
print("  Sending {} spans (from {} raw) to the recommendation service.".format(len(refund_spans), raw_count))

# Figure out which tools were actually called by inspecting the spans we just
# downloaded (no extra CloudWatch query needed).
invoked_names = set()
for span in refund_spans:
    attrs = span.get("attributes", {}) if isinstance(span, dict) else {}
    tname = attrs.get("gen_ai.tool.name") if isinstance(attrs, dict) else None
    if tname:
        invoked_names.add(tname)

configured_names = {t["toolName"] for t in REFUND_AGENT_TOOLS}
target_names = configured_names & invoked_names
unused_names = configured_names - invoked_names

print("\nRefund Agent tools:      {}".format(sorted(configured_names)))
print("Called in our sessions:  {}".format(sorted(invoked_names) if invoked_names else "(none detected)"))
if unused_names:
    print("Skipping (not invoked):  {}".format(sorted(unused_names)))

if not target_names or not refund_spans:
    print("\nSkipping tool-description recommendation \u2014 no invoked tools found in fresh spans.")
    TOOL_REC_ID = None
else:
    tools_for_api = [
        {"toolName": t["toolName"], "toolDescription": {"text": t["toolDescription"]}}
        for t in REFUND_AGENT_TOOLS if t["toolName"] in target_names
    ]
    print("\nAsking for improved descriptions for {} tool(s): {}".format(
        len(tools_for_api), sorted(t["toolName"] for t in tools_for_api)))

    resp = agentcore.start_recommendation(
        name="refund_tooldesc_{}".format(uuid.uuid4().hex[:6]),
        description="Sharpen the Refund Agent's tool descriptions.",
        type="TOOL_DESCRIPTION_RECOMMENDATION",
        recommendationConfig={
            "toolDescriptionRecommendationConfig": {
                "toolDescription": {"toolDescriptionText": {"tools": tools_for_api}},
                "agentTraces": {"sessionSpans": refund_spans},
            }
        },
    )
    TOOL_REC_ID = resp["recommendationId"]
    print("Started: {}".format(TOOL_REC_ID))
    result = poll_recommendation(TOOL_REC_ID)
    tr = (result.get("recommendationResult") or {}).get("toolDescriptionRecommendationResult") or {}

    if tr.get("errorCode"):
        print("\nERROR: {} \u2014 {}".format(tr.get("errorCode"), tr.get("errorMessage")))
    else:
        print("\n" + "=" * 20 + " RECOMMENDATIONS " + "=" * 20)
        for t in tr.get("tools", []):
            current = next((td["toolDescription"] for td in REFUND_AGENT_TOOLS if td["toolName"] == t["toolName"]), "")
            print("\nTool: {}".format(t["toolName"]))
            print("  Current description:")
            print("    " + current.replace("\n", "\n    ")[:400])
            print("\n  Recommended description:")
            print("    " + (t.get("recommendedToolDescription") or "").replace("\n", "\n    ")[:400])
            if t.get("explanation"):
                print("\n  Why this change:")
                print("    " + t["explanation"].replace("\n", "\n    ")[:400])

## Step 5: Save the current settings as version 1
A configuration bundle is a versioned snapshot of the settings of your agent, such as its system prompt, model and tools. Every save creates a new frozen version that you can go back to.

This step creates the bundle and saves the current Harness settings as version 1. Your live Harness does not change.

In [ ]:
BUNDLE_NAME = "anycompany_harness_config"

# Grab the Harness's current model config too, so the snapshot is complete.
# L3 creates the Harness with liteLlmModelConfig; fall back to bedrockModelConfig.
_harness_model_cfg = harness_state.get("model", {}) or {}
harness_model = _harness_model_cfg.get("liteLlmModelConfig") or _harness_model_cfg.get("bedrockModelConfig") or {}

# Convention: the bundle's component key is the ARN of the resource we're describing.
bundle_components = {
    cfg["harness_arn"]: {
        "configuration": {k: v for k, v in {
            "systemPrompt": CURRENT_SYSTEM_PROMPT,
            "modelId": harness_model.get("modelId"),
            "temperature": harness_model.get("temperature"),   # omitted when the Harness sets none
            "maxTokens": harness_model.get("maxTokens"),
        }.items() if v is not None}
    }
}

try:
    resp = agentcore_control.create_configuration_bundle(
        bundleName=BUNDLE_NAME,
        description="AnyCompany orchestrator Harness configuration snapshot.",
        components=bundle_components,
        branchName="mainline",
        commitMessage="Initial snapshot of the deployed Harness configuration.",
    )
    BUNDLE_ID = resp["bundleId"]
    BUNDLE_ARN = resp["bundleArn"]
    INITIAL_VERSION_ID = resp["versionId"]
    print("Created bundle version 1:")
    print("  bundleId:  {}".format(BUNDLE_ID))
    print("  versionId: {}".format(INITIAL_VERSION_ID))
except agentcore_control.exceptions.ConflictException:
    BUNDLE_ID = None
    for b in agentcore_control.list_configuration_bundles().get("bundles", []):
        if b["bundleName"] == BUNDLE_NAME:
            BUNDLE_ID = b["bundleId"]
            BUNDLE_ARN = b["bundleArn"]
            break
    if not BUNDLE_ID:
        raise
    got = agentcore_control.get_configuration_bundle(bundleId=BUNDLE_ID)
    INITIAL_VERSION_ID = got["versionId"]
    print("Bundle already exists, reusing:")
    print("  bundleId:  {}".format(BUNDLE_ID))
    print("  latest versionId: {}".format(INITIAL_VERSION_ID))

## Step 6: Save the recommended prompt as version 2
Save the recommended prompt from Step 3 as version 2 of the same bundle, linked to version 1 as its parent. Version 1 stays as it was. Version 2 is what you would use if you chose to apply the recommendation.

In [ ]:
if not RECOMMENDED_SYSTEM_PROMPT:
    print("Skipping \u2014 Step 3 didn't produce a recommendation.")
    NEW_VERSION_ID = None
else:
    # Same components as version 1, but with the recommended prompt swapped in.
    new_components = {
        cfg["harness_arn"]: {
            "configuration": {k: v for k, v in {
                "systemPrompt": RECOMMENDED_SYSTEM_PROMPT,
                "modelId": harness_model.get("modelId"),
                "temperature": harness_model.get("temperature"),   # omitted when the Harness sets none
                "maxTokens": harness_model.get("maxTokens"),
            }.items() if v is not None}
        }
    }
    resp = agentcore_control.update_configuration_bundle(
        bundleId=BUNDLE_ID,
        components=new_components,
        parentVersionIds=[INITIAL_VERSION_ID],
        branchName="mainline",
        commitMessage="Applied prompt recommendation from Step 3.",
    )
    NEW_VERSION_ID = resp["versionId"]
    print("Saved bundle version 2:")
    print("  versionId: {}".format(NEW_VERSION_ID))
    print("  parent:    {}".format(INITIAL_VERSION_ID))

## Step 7: Look at the version history
List every version of the bundle. Each one records its parent, who created it and a commit message, which gives you a full history of your prompt changes.

In [ ]:
versions_resp = agentcore_control.list_configuration_bundle_versions(bundleId=BUNDLE_ID)
rows = []
for v in versions_resp.get("versions", []):
    lm = v.get("lineageMetadata", {})
    rows.append({
        "versionId": v["versionId"],
        "branch": lm.get("branchName"),
        "parents": ",".join(lm.get("parentVersionIds") or []) or "(root)",
        "author": (lm.get("createdBy") or {}).get("name") or "(system)",
        "message": (lm.get("commitMessage") or "")[:60],
        "createdAt": str(v.get("versionCreatedAt", ""))[:19],
    })

df = pd.DataFrame(rows)
print("Version history for bundle '{}':\n".format(BUNDLE_NAME))
print(df.to_string(index=False))

## Summary

This notebook asked AgentCore for a better system prompt and better tool descriptions, based on your agents' traces, and saved the current and recommended prompts as two versions of a configuration bundle so you can see the history of changes. Nothing was applied to the running agents.

**Next:** This is the last notebook. When you are done with the workshop, run the Cleanup cell at the end of each notebook, last notebook first.

## Step 8: Clean up
Delete the recommendations and the bundle created here. The IAM role stays in place because the evaluations notebook also uses it.

In [ ]:
## === CLEANUP: Delete all resources created in this notebook ===
## \u26a0\ufe0f Uncomment and run this cell only after you are done with ALL notebooks.

# # Delete the two recommendations (TOOL_REC_ID may be None if Step 4 skipped)
# for rec_id, label in [(SYSPROMPT_REC_ID, "system-prompt"),
#                        (TOOL_REC_ID, "tool-description")]:
#     if not rec_id:
#         print("Skipping ({}) \u2014 no recommendation was created.".format(label))
#         continue
#     print("Deleting recommendation ({})...".format(label))
#     try:
#         agentcore.delete_recommendation(recommendationId=rec_id)
#         print("  OK")
#     except Exception as e:
#         print("  {}".format(e))
#
# # Delete the bundle
# print("\nDeleting configuration bundle...")
# try:
#     agentcore_control.delete_configuration_bundle(bundleId=BUNDLE_ID)
#     print("  OK")
# except Exception as e:
#     print("  {}".format(e))
#
# print("\nDone. The IAM role '{}' is left in place (shared with L7 notebook 1).".format(OPT_ROLE_NAME))